# Exercise: Classify Clothing Images with a CNN

[Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) has the same format as MNIST: 70,000 greyscale images of 28 × 28 pixels in ten classes. The images show clothing and shoes from the Zalando online shop instead of digits, and they are harder to classify: a shirt, a T-shirt and a pullover look much alike at this resolution.

In this exercise you build a dense baseline and a convolutional network, diagnose their learning curves, reduce overfitting with dropout and early stopping, and evaluate your chosen model once on the test set. The tasks describe what to build and what to report; the code is yours to write. Notebooks 01 and 02 contain every layer and function you need. A CNN trains for about two minutes, so plan your runs before you start them.

## Learning Objectives

By the end of this exercise, you should be able to:

- Build a dense baseline and a CNN for Fashion-MNIST and compare them on a validation set.
- Calculate the output shapes and parameters of a CNN and check them against `model.summary()`.
- Diagnose overfitting in the learning curves of a CNN and reduce it with dropout and early stopping.
- Evaluate the chosen model on the test set and identify the classes it confuses most.

## Setup

In [ ]:
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)

## Step 1: Load the Data

[`keras.datasets.fashion_mnist.load_data()`](https://keras.io/api/datasets/fashion_mnist/) returns 60,000 training and 10,000 test images and their labels from 0 to 9; it downloads them on the first run. The cell below loads the data and shows one example of each class.

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]
print("Training images:", X_train_full.shape, " test images:", X_test.shape)

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for label, ax in enumerate(axes):
    ax.imshow(X_train_full[y_train_full == label][0], cmap="gray")
    ax.set_title(class_names[label], fontsize=9)
    ax.axis("off")
plt.show()

## Step 2: Prepare the Data

- Split the training images into a training set and a validation set of 10,000 images, with `stratify` and `random_state=42`.
- Add a channel axis to the training, validation and test images, so each set has the shape `(images, 28, 28, 1)`.
- Print the shapes of the three sets.

Scale the pixels inside every model with a `Rescaling(1 / 255)` layer, as in notebook 02.

## Step 3: Plan How You Train

You will train four or five models. How you organise the code is up to you:

- **Write helper functions** (optional), similar to those in notebook 02. For example, `compile_and_fit(model, epochs, callbacks=None)` compiles and trains a model and returns the history as a DataFrame, and `plot_curves(logs, title)` draws the loss next to the accuracy.
- **Write each model out in full**, and copy and change the cells from one step to the next.

Whichever way you choose, keep these settings the same for every model, so the comparison is fair and your numbers are close to the solution's:

- Call `keras.utils.clear_session()` and `keras.utils.set_random_seed(42)` before you build each model.
- Compile with Adam (learning rate 0.001), the `sparse_categorical_crossentropy` loss and the accuracy metric.
- Train with `batch_size=128` and the validation set as `validation_data`.
- Plot the loss and the accuracy curves of every run, for both the training and the validation set.

Also create an empty dictionary `results`. After every model, store its validation accuracy there under a descriptive name, so you can compare all models in step 8.

The cell below is for your helper functions, if you write them; you can also leave it empty.

## Step 4: A Dense Baseline

Build a dense network: the input, a `Rescaling` layer, `Flatten`, one hidden layer of 128 ReLU neurons and a softmax output layer with one neuron per class. Train it for 20 epochs, plot the learning curves and store the validation accuracy.

How many parameters does it have? Does it overfit?

*Your answers:*

## Step 5: A Convolutional Network

Build a CNN with two convolutional blocks:

1. `Conv2D` with 32 filters of 3 × 3 and ReLU, followed by `MaxPooling2D` with `pool_size=2`,
2. `Conv2D` with 64 filters of 3 × 3 and ReLU, followed by `MaxPooling2D` with `pool_size=2`,

then `Flatten`, a dense layer of 128 ReLU neurons and the softmax output layer. Use the default `padding="valid"`.

**Before you train it**, calculate the output shape and the number of parameters of every layer with the formulas of notebook 01, and write them in the Markdown cell below. Remember that pooling rounds down: an odd size such as 11 becomes 5. Then print `model.summary()` and compare. Which layer has the most parameters?

Train the CNN for 15 epochs, plot the learning curves and store the validation accuracy. At which epoch is the validation loss lowest, and what happens afterwards?

*Your calculation (layer, output shape, parameters):*

*Your answers:*

## Step 6: Add Dropout

Build the same CNN with a `Dropout` layer with a rate of 0.3 after `Flatten` and another after the dense hidden layer. Train it for 15 epochs, plot the learning curves and store the validation accuracy.

Compare the curves with those of step 5: has the gap between training and validation become smaller?

*Your answers:*

## Step 7: Train Longer and Keep the Best Epoch

Train the CNN with dropout again, this time for up to 30 epochs, with two callbacks:

- [`EarlyStopping`](https://keras.io/api/callbacks/early_stopping/) that monitors the validation accuracy with `patience=5` and `restore_best_weights=True`,
- [`ModelCheckpoint`](https://keras.io/api/callbacks/model_checkpoint/) that saves the best model to `models/fashion_cnn_best.keras`, monitoring the same value.

Create the `models` folder first. Load the saved model with [`keras.saving.load_model()`](https://keras.io/api/models/model_saving_apis/model_saving_and_loading/#loadmodel-function), store its validation accuracy and plot the learning curves. After how many epochs did training stop?

*Your answers:*

## Step 8: Compare the Models

Show the validation accuracies of all your models in one table, sorted from best to worst, and choose your final model.

## Step 9: Evaluate Once on the Test Set

Evaluate your chosen model on the test set. The choice is already made, so also report the dense baseline's test accuracy for reference only. Then plot the confusion matrix of your chosen model on the test set with the class names as labels, and show a few misclassified test images with their true and predicted class.

Which classes does your model confuse most? Looking at the example images in step 1, why could that be?

*Your answers:*

## Reflection

Answer these questions in the Markdown cell below.

1. Compare the parameter counts of the dense network and the CNN, layer by layer. Where are most of the CNN's parameters, and why?
2. Did the CNN without dropout overfit? Which signs in the learning curves show it, and what did dropout change?
3. Your final model still confuses some classes. Would a larger CNN solve this, or is part of the problem in the data itself?
4. Every model was compared on the validation set, and the test set only reported the accuracy of the chosen model. What would go wrong if you had chosen the model by its test accuracy?

*Your answers:*

## References

- Xiao, H., Rasul, K. and Vollgraf, R. (2017). [Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms](https://arxiv.org/abs/1708.07747).
- [Fashion-MNIST on GitHub](https://github.com/zalandoresearch/fashion-mnist): the dataset, its classes and a benchmark of many models.
- TensorFlow tutorial: [Basic classification: Classify images of clothing](https://www.tensorflow.org/tutorials/keras/classification), a dense network on the same data.
- Keras API: [`Conv2D`](https://keras.io/api/layers/convolution_layers/convolution2d/), [`Dropout`](https://keras.io/api/layers/regularization_layers/dropout/), [`EarlyStopping`](https://keras.io/api/callbacks/early_stopping/) and [`ModelCheckpoint`](https://keras.io/api/callbacks/model_checkpoint/).